# Chapter 17 — Search the Reasoning Space

**Book alignment:** DSPy From First Principles, Chapter 17

**Question this notebook isolates:** Does memoizing expansion on (state, trace) collapse three sibling expansions into one repeated child?


In [ ]:
from pathlib import Path
import math
import random
import sys

random.seed(0)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy


class TraceStep(dspy.Signature):
    """Produce the next reasoning step given what is known."""

    context: str = dspy.InputField()
    trace: str = dspy.InputField()
    reasoning: str = dspy.OutputField()


class ScoreReasoning(dspy.Signature):
    """Evaluate how promising a reasoning path is."""

    context: str = dspy.InputField()
    trace: str = dspy.InputField()
    score: str = dspy.OutputField()


reasoning_fn = dspy.Predict(TraceStep)
score_fn = dspy.Predict(ScoreReasoning)
print("dspy", dspy.__version__, "| generation and value signatures constructed, never executed")


## Selection spends finite compute

UCT balances exploitation against exploration: an unvisited child must be tried first, and an under-explored child can outrank a high-average leader. The fixture mirrors the appendix table.


In [ ]:
def uct_value(visits, reward, parent_visits, ucb_weight=1.41):
    if visits == 0:
        return float("inf")
    exploitation = reward / max(1, visits)
    exploration = ucb_weight * math.sqrt(max(1e-9, math.log(max(1, parent_visits)) / visits))
    return exploitation + exploration

PARENT_VISITS = 13
children = [
    {"id": "unvisited", "visits": 0, "reward": 0.0},
    {"id": "leader", "visits": 10, "reward": 7.0},
    {"id": "underexplored", "visits": 3, "reward": 0.9},
]
for child in children:
    child["uct"] = uct_value(child["visits"], child["reward"], PARENT_VISITS)
    avg = child["reward"] / max(1, child["visits"])
    print(f"{child['id']:14s} avg={avg:.2f} uct={child['uct']:.4f}")
order = [c["id"] for c in sorted(children, key=lambda c: c["uct"], reverse=True)]
print("selection order:", order)


In [ ]:
by_id = {c["id"]: c for c in children}
assert order == ["unvisited", "underexplored", "leader"]
assert by_id["underexplored"]["uct"] > by_id["leader"]["uct"]
assert (by_id["underexplored"]["reward"] / 3) < (by_id["leader"]["reward"] / 10)
print("exploration term does real work: lower average outranks the leader")


## Cache collapses branching

Expansion asks for *another* candidate continuation from the same parent. A cache keyed on identical (state, trace) answers that request with the first continuation every time, so three children share one trace.


In [ ]:
class FakeGenerator:
    def __init__(self):
        self.calls = 0
        self.cache = {}

    def cached_expand(self, state, trace):
        key = (state, tuple(trace))
        if key in self.cache:
            return self.cache[key]
        self.calls += 1
        step = f"thought-{self.calls}"
        self.cache[key] = step
        return step

    def fresh_expand(self, state, trace):
        self.calls += 1
        return f"thought-{self.calls}"

gen = FakeGenerator()
parent_trace: list = []
cached_children = [gen.cached_expand("evidence", parent_trace) for _ in range(3)]
gen2 = FakeGenerator()
fresh_children = [gen2.fresh_expand("evidence", parent_trace) for _ in range(3)]
print("cached siblings:", cached_children, "unique:", len(set(cached_children)))
print("fresh siblings: ", fresh_children, "unique:", len(set(fresh_children)))


In [ ]:
assert len(set(cached_children)) == 1
assert len(set(fresh_children)) == 3
print("drawing a tree does not mean exploring one: report unique_traces, not calls")


## Budget must count all calls; oracle guidance is invalid

Generation, scoring, reflection, and synthesis are all model invocations. And a search-time scorer allowed to see the reference answer steers toward a handed target, so its result is discarded on principle.


In [ ]:
cost = {"generation": 12, "scoring": 12, "reflection": 5, "synthesis": 1}
total_calls = sum(cost.values())
oracle_scorer_inputs = {"context", "trace", "known_answer"}
fair_scorer_inputs = {"context", "trace"}
oracle_invalid = "known_answer" in oracle_scorer_inputs
fair_invalid = "known_answer" in fair_scorer_inputs
print("call counts:", cost, "-> total:", total_calls, "vs generator-only:", cost["generation"])
print("oracle scorer sees answer:", oracle_invalid, "| fair scorer sees answer:", fair_invalid)


In [ ]:
assert total_calls == 30
assert total_calls != cost["generation"]
assert oracle_invalid is True
assert fair_invalid is False
print("compare quality against total inference cost; discard oracle arms by construction")


## What we earned

Search does not know truth; search knows reward. The audit shows why code that looks like search must be read as a causal graph: the cache can erase branching, the budget must count every call, and only a consumed value function steers anything.

Notebook 18 / Chapter 18 closes that boundary across the whole optimization loop with an experimental firewall.
